# Raikou Optical · Phase 0 · Notebook 2 — fine-tune OlmoEarth v1.2 Base

**Runtime: GPU → L4** (A100 also works and is about 3× faster per hour but costs about 3× the units). *Runtime → Change runtime type → L4 GPU, High-RAM on.*

Model: OlmoEarth v1.2 Base encoder (114M) with LLRD 0.65 → masked temporal attention pooling → 10 m spectral stem from the clearest date → progressive U-Net decoder (GroupNorm) → 5 classes. Loss 0.5 class-weighted CE + 0.5 Dice, ignoring 255. bf16. Random timestep dropout (T = 1/2/3 with p = 0.4/0.2/0.4) so single-date runs work.

**Resumable**: `last_epoch.pt` is saved to Drive after every epoch. After a disconnect, reconnect, run steps 1–5 again, then step 7; it continues after the last finished epoch.

## 1 · Mount Drive, check the GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, json
ROOT = '/content/drive/MyDrive/Raikou_Optical_Phase0'   # change only if you put the folder elsewhere
CODE = f'{ROOT}/optical_phase0'
assert os.path.exists(f'{CODE}/raikou_phase0/__init__.py'), f'Upload the optical_phase0 folder into {ROOT} first (see README).'
os.environ['RAIKOU_P0_ROOT'] = ROOT
if CODE not in sys.path:
    sys.path.insert(0, CODE)

import torch
!nvidia-smi --query-gpu=name,memory.total --format=csv
print('torch', torch.__version__, '| CUDA', torch.version.cuda, '| bf16', torch.cuda.is_available() and torch.cuda.is_bf16_supported())

## 2 · Install the OlmoEarth loader (keeps Colab's own torch)

In [ ]:
v = tuple(int(x) for x in torch.__version__.split('+')[0].split('.')[:2])
assert v >= (2, 7), f'Colab torch {torch.__version__} is older than 2.7, which the OlmoEarth package needs'
!pip -q install olmoearth-pretrain-minimal==0.1.2
# Optional mirror instead of Hugging Face (same bytes, SHA-256 checked either way):
# !pip -q install modelscope

## 3 · Settings and the base weights
Edit `TrainConfig(...)` here if you want something else. `patch_size=2` needs `micro_batch=4, grad_checkpointing=True` and costs roughly 6–8× the compute; the speed probe in step 6 tells you exactly. The base files are refused unless their SHA-256 matches the official release.

In [ ]:
from tqdm.auto import tqdm
from raikou_phase0 import config as C, tiles, data as D, model as Mdl, train as TR, selftest
paths = C.Paths().ensure()
tile_list = tiles.load_manifest(paths.manifest_csv)
train_ids = [t.tile_id for t in tile_list if t.split == 'train']
val_ids = [t.tile_id for t in tile_list if t.split == 'val']
cfg = C.TrainConfig()            # e.g. C.TrainConfig(patch_size=2, micro_batch=4, grad_checkpointing=True)
print(len(train_ids), 'train /', len(val_ids), 'val tiles')
print(json.dumps(cfg.to_dict(), indent=1))
base = Mdl.fetch_base_files(cfg.weights_source)   # 'huggingface' (default) or 'modelscope'
print('OlmoEarth v1.2 Base verified:', base['weights_sha256'][:16], '...')

## 4 · GPU self-test (under a minute)
Forward passes with T = 1, 2, 3 and cloud masks, the eval-batching padding check, and one backward pass.

In [ ]:
selftest.gpu_selftest(base['base_dir'], cfg.patch_size)

## 5 · Copy the tiles to the VM's local disk (a few minutes)

In [ ]:
missing = [t for t in train_ids + val_ids if not os.path.exists(f'{paths.tiles}/{t}.npz')]
assert not missing, f'{len(missing)} tiles are not built yet; finish notebook 1 first: {missing[:5]}'
D.stage_tiles(train_ids + val_ids, paths, progress=tqdm)
!du -sh /content/local_dataset

## 6 · Speed probe (about a minute, saves nothing)
Runs 20 real steps and projects the whole run. Type the compute-unit rate Colab shows for this GPU (*Runtime → View resources*) into `CU_PER_HOUR` to see the cost.

In [ ]:
probe = TR.speed_probe(cfg, paths, train_ids, val_ids, base, steps=20)
CU_PER_HOUR = None   # e.g. 1.7 - read it from Colab, it changes
hours = probe['hours_for_all_epochs'] * 1.1   # +10% for validation and checkpoint copies
print(f'Projected: {hours:.1f} GPU hours' + (f' = {hours * CU_PER_HOUR:.0f} compute units' if CU_PER_HOUR else ''))

## 7 · Train (hours; resumable)
Each epoch prints one JSON line: train loss, val mIoU at T=1 and T=3 against cleaned and raw labels, and whether it saved a new best. Keep the tab open; Colab may stop idle sessions.

In [ ]:
history = TR.train(cfg, paths, train_ids, val_ids, base, progress=tqdm)

## 8 · Curves and the best checkpoint

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
h = pd.read_csv(f'{paths.checkpoints}/train_log.csv')
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
h.plot(x='epoch', y='train_loss', ax=ax[0], legend=False, title='train loss')
cols = [c for c in h.columns if c.endswith('_miou')]
h.plot(x='epoch', y=cols, ax=ax[1], title='validation mIoU'); plt.show()
best = torch.load(f'{paths.checkpoints}/best_model.pt', map_location='cpu', weights_only=True)
print('best epoch:', best['epoch'])
print(json.dumps({m: {k: best['val'][m][k]['mean_iou'] for k in ('clean', 'raw')} for m in ('T1', 'T3')}, indent=1))

Done. Next: **Notebook 3** on a T4 GPU runtime.